# 06 Agent loop
A minimal agent: two tools, a JSON tool protocol, and two stop rules.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork (and BRANCH).
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
BRANCH = "main"  # change to your branch name if you are working on a branch
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-06"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: the two tools

In [ ]:
import json
import logging
from mcp_server.server import search

for name in ("httpx", "httpx2"):  # the MCP library turns on one log line per HTTP call; hide them
    logging.getLogger(name).setLevel(logging.WARNING)

OUTBOX = OUTPUT / "outbox.jsonl"

def search_docs(query, top_k=3):
    return search(query, max(1, min(int(top_k), 10)), CORPUS)

def draft_issue(title, body):
    # Writes ONLY to a local file. Nothing is sent anywhere.
    with open(OUTBOX, "a") as f:
        f.write(json.dumps({"title": title, "body": body}) + "\n")
    return {"saved": True, "outbox": OUTBOX.name}

TOOLS = {"search_docs": search_docs, "draft_issue": draft_issue}
print([h["id"] for h in search_docs("export a note as pdf")])

Step 3: the tool protocol, written in the prompt

In [ ]:
import json
from kit.llm import call_llm

# The tool protocol lives in the prompt. Gemini and OpenAI both have native tool calling, but in
# different formats; plain JSON in text works the same with every provider and with the mock.
SYSTEM = """You are an agent for the Quillnote help desk.
TOOL PROTOCOL: reply with exactly one JSON object and nothing else.
To call a tool: {"tool": "<name>", "args": {<arguments>}}
To finish:      {"final": "<answer for the user, citing page ids like [qn-02]>"}
Tools:
- search_docs(query: str, top_k: int = 3): search the Quillnote help pages. Read-only.
- draft_issue(title: str, body: str): save a draft bug report to the local outbox. Use it only when the task asks for an issue.
Every reply is JSON, your final answer too. Never reply with plain text.
Call one tool per reply. Finish as soon as you can answer."""

def build_prompt(task, steps):
    lines = [f"Task: {task}", "", "Steps so far:"]
    for n, s in enumerate(steps, 1):
        lines += [f"CALL {n}: {json.dumps(s['call'])}", f"RESULT {n}: {s['result']}"]
    if not steps:
        lines.append("(none yet)")
    return "\n".join(lines + ["", "Reply with your next JSON object."])

def parse_action(text):
    # Take the outermost {...}: models sometimes wrap the JSON in ``` fences or add a sentence.
    try:
        action = json.loads(text[text.find("{"): text.rfind("}") + 1])
        return action if isinstance(action, dict) else None
    except ValueError:
        return None

print(build_prompt("How do I export a note as PDF?", []))

Step 4: the loop: about 40 lines

In [ ]:
def run_agent(task, tools, max_steps=6):
    steps = []
    for n in range(1, max_steps + 1):
        reply = call_llm(build_prompt(task, steps), system=SYSTEM, max_output_tokens=1000)
        action = parse_action(reply["text"])
        if action is None:
            action, result = {"raw": reply["text"][:200]}, 'ERROR: reply was not one JSON object. Reply with {"final": "..."} or a tool call.'
        elif "final" in action:
            print(f"  step {n}: final answer")
            return {"stop": "final_answer", "answer": action["final"], "steps": steps}
        elif action.get("tool") not in tools:
            result = f"ERROR: unknown tool {action.get('tool')}"
        else:
            try:
                result = json.dumps(tools[action["tool"]](**action.get("args", {})))
            except Exception as e:  # a broken tool becomes an observation, not a crash
                result = f"ERROR: {type(e).__name__}: {e}"
        steps.append({"call": action, "result": result[:2000]})
        print(f"  step {n}: {json.dumps(action)[:100]} -> {result[:80]}")
    return {"stop": "max_steps", "answer": None, "steps": steps}

Step 5: two tasks that should work

In [ ]:
OUTBOX.unlink(missing_ok=True)
report = []
for task in ["How do I export a note as PDF?",
             'PDF export cuts off wide tables. Draft an issue titled "PDF export cuts off tables" for the docs team.']:
    print("Task:", task)
    out = run_agent(task, TOOLS)
    print("Stop:", out["stop"], "| answer:", out["answer"], "\n")
    report.append({"run": task[:40], "stop": out["stop"], "tool_steps": len(out["steps"])})
print("Outbox lines:", len(OUTBOX.read_text().splitlines()) if OUTBOX.exists() else 0)

Step 6: break it: a tool that always errors

In [ ]:
def broken_search(query, top_k=3):
    raise ConnectionError("search service is down")

out = run_agent("How do I export a note as PDF?", {**TOOLS, "search_docs": broken_search})
print("Stop that fired:", out["stop"], "| answer:", out["answer"])
report.append({"run": "erroring tool", "stop": out["stop"], "tool_steps": len(out["steps"])})

Step 7: break it: a task that never ends

In [ ]:
task = 'Keep calling search_docs with "teleport to mars" until a result mentions Mars, then answer.'
out = run_agent(task, TOOLS, max_steps=5)
print("Stop that fired:", out["stop"], "| steps used:", len(out["steps"]))
report.append({"run": "never-ending task", "stop": out["stop"], "tool_steps": len(out["steps"])})

Step 8: which stop fired, run by run

In [ ]:
import pandas as pd
pd.DataFrame(report)